# Karaoke Pitch & Alignment Validation
## Using Kiritan Singing Database as Ground Truth

**Objectives:**
1. **Alignment:** Is Qwen3ForcedAligner better than Whisper's built-in timestamps?
2. **Pitch:** Is FCPE accurate enough for singing voice pitch tracking?
3. **Scoring:** Calibrate forgiving thresholds so pipeline errors don't punish singers.

**Ground Truth:**
- `mono_label/*.lab` — phoneme-level timing (hand-corrected)
- `midi_label/*.mid` — note-level pitch (score transcription)

**Philosophy:** Forgiving scoring. The pipeline is procedurally generated and will make mistakes. We'd rather give the singer free points than punish them for model faults.

In [29]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import torch
import matplotlib.pyplot as plt
import mido
from pathlib import Path
from collections import defaultdict

# Project modules
from tools.pitch import (
    PitchExtractor, hz_to_midi, midi_to_hz, midi_to_note_name,
    clean_pitch_curve, assign_pitch_to_words, build_pitch_section,
    HOP_SECONDS, FCPE_SAMPLE_RATE
)
from tools.transcription import TranscriptionEngine
from qwen_asr import Qwen3ForcedAligner

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Paths
KIRITAN_DIR = Path("../TestData/kiritan_singing")
WAV_DIR = KIRITAN_DIR / "wav"
LABEL_DIR = KIRITAN_DIR / "mono_label"
MIDI_DIR = KIRITAN_DIR / "midi_label"

# Use first 5 samples for quick iteration
SAMPLE_IDS = ["01", "02", "03", "04", "05"]

print(f"Kiritan dir exists: {KIRITAN_DIR.exists()}")
print(f"Available WAVs: {len(list(WAV_DIR.glob('*.wav')))}")
print(f"Available labels: {len(list(LABEL_DIR.glob('*.lab')))}")
print(f"Available MIDIs: {len(list(MIDI_DIR.glob('*.mid')))}")

Device: cuda
Kiritan dir exists: True
Available WAVs: 50
Available labels: 50
Available MIDIs: 50


In [32]:
# Check environment and WhisperX
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

try:
    import whisperx
    print("✓ WhisperX available")
except ImportError as e:
    print(f"✗ WhisperX not available: {e}")


Python executable: /home/august/PythonUtilities/ProcessVocalDataset/.venv/bin/python
Python version: 3.10.19 (main, Jan 27 2026, 23:59:05) [Clang 21.1.4 ]
✗ WhisperX not available: No module named 'whisperx'


## Section 2: Load Kiritan Ground Truth

The kiritan dataset provides:
- **Phoneme labels** (`.lab`): `start_sec end_sec phoneme` — hand-corrected alignment truth
- **MIDI labels** (`.mid`): Note events with pitch and timing — pitch truth

We'll parse both into usable Python structures.

In [17]:
# ── Parse phoneme labels ──────────────────────────────────────────────
def parse_phoneme_labels(lab_path: Path) -> list[dict]:
    """Parse kiritan .lab file → list of {start, end, phoneme}."""
    entries = []
    for line in lab_path.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) == 3:
            entries.append({
                "start": float(parts[0]),
                "end": float(parts[1]),
                "phoneme": parts[2],
            })
    return entries


def phonemes_to_words(phonemes: list[dict], table_path: Path = None) -> list[dict]:
    """
    Group phonemes into approximate 'word' segments.
    Words are delimited by 'pau' (pause) or 'br' (breath) phonemes.
    Returns list of {start, end, text} where text is the phoneme sequence.
    """
    DELIMITERS = {"pau", "br", "cl", "sil"}  # cl = closure, often a word boundary
    words = []
    current_phonemes = []
    word_start = None

    for p in phonemes:
        if p["phoneme"] in DELIMITERS:
            if current_phonemes:
                words.append({
                    "start": word_start,
                    "end": current_phonemes[-1]["end"],
                    "text": "".join(pp["phoneme"] for pp in current_phonemes),
                    "phonemes": current_phonemes,
                })
                current_phonemes = []
                word_start = None
        else:
            if word_start is None:
                word_start = p["start"]
            current_phonemes.append(p)

    if current_phonemes:
        words.append({
            "start": word_start,
            "end": current_phonemes[-1]["end"],
            "text": "".join(pp["phoneme"] for pp in current_phonemes),
            "phonemes": current_phonemes,
        })
    return words


# ── Parse MIDI labels ────────────────────────────────────────────────
def parse_midi_notes(mid_path: Path) -> list[dict]:
    """
    Parse kiritan MIDI file → list of {start_sec, end_sec, midi_note, hz}.
    These are the ground truth sung pitches (from the score).
    """
    mid = mido.MidiFile(str(mid_path))
    ticks_per_beat = mid.ticks_per_beat

    # Find tempo
    tempo = 500000  # default 120 BPM
    for track in mid.tracks:
        for msg in track:
            if msg.type == "set_tempo":
                tempo = msg.tempo
                break

    sec_per_tick = tempo / (ticks_per_beat * 1_000_000)

    notes = []
    active = {}  # note_num -> start_time_sec
    abs_tick = 0

    # Use track 1 (MIDI track with note events)
    for track in mid.tracks:
        abs_tick = 0
        for msg in track:
            abs_tick += msg.time
            t_sec = abs_tick * sec_per_tick

            if msg.type == "note_on" and msg.velocity > 0:
                active[msg.note] = t_sec
            elif msg.type == "note_on" and msg.velocity == 0:
                if msg.note in active:
                    start = active.pop(msg.note)
                    hz = 440.0 * 2 ** ((msg.note - 69) / 12.0)
                    notes.append({
                        "start_sec": round(start, 4),
                        "end_sec": round(t_sec, 4),
                        "midi_note": msg.note,
                        "hz": round(hz, 2),
                    })
            elif msg.type == "note_off":
                if msg.note in active:
                    start = active.pop(msg.note)
                    hz = 440.0 * 2 ** ((msg.note - 69) / 12.0)
                    notes.append({
                        "start_sec": round(start, 4),
                        "end_sec": round(t_sec, 4),
                        "midi_note": msg.note,
                        "hz": round(hz, 2),
                    })

    notes.sort(key=lambda n: n["start_sec"])
    return notes


# ── Load all samples ──────────────────────────────────────────────────
gt_data = {}  # sample_id -> {phonemes, words, midi_notes, wav_path}

for sid in SAMPLE_IDS:
    wav_path = WAV_DIR / f"{sid}.wav"
    lab_path = LABEL_DIR / f"{sid}.lab"
    mid_path = MIDI_DIR / f"{sid}.mid"

    phonemes = parse_phoneme_labels(lab_path)
    words = phonemes_to_words(phonemes)
    midi_notes = parse_midi_notes(mid_path)

    gt_data[sid] = {
        "wav_path": str(wav_path),
        "phonemes": phonemes,
        "words": words,
        "midi_notes": midi_notes,
    }

    print(f"Sample {sid}: {len(phonemes)} phonemes, {len(words)} word-groups, "
          f"{len(midi_notes)} MIDI notes")

# Preview first sample's MIDI notes
print("\n--- Sample 01: First 10 MIDI notes ---")
for n in gt_data["01"]["midi_notes"][:10]:
    print(f"  {n['start_sec']:.3f}-{n['end_sec']:.3f}s  "
          f"MIDI {n['midi_note']} ({midi_to_note_name(n['midi_note'])})  "
          f"{n['hz']:.1f} Hz")

Sample 01: 444 phonemes, 41 word-groups, 202 MIDI notes
Sample 02: 352 phonemes, 29 word-groups, 175 MIDI notes
Sample 03: 296 phonemes, 25 word-groups, 143 MIDI notes
Sample 04: 587 phonemes, 46 word-groups, 282 MIDI notes
Sample 05: 502 phonemes, 32 word-groups, 250 MIDI notes

--- Sample 01: First 10 MIDI notes ---
  18.947-19.351s  MIDI 64 (E4)  329.6 Hz
  19.342-21.904s  MIDI 71 (B4)  493.9 Hz
  22.105-22.302s  MIDI 68 (G#4)  415.3 Hz
  22.309-22.492s  MIDI 69 (A4)  440.0 Hz
  22.500-22.895s  MIDI 71 (B4)  493.9 Hz
  23.290-23.684s  MIDI 71 (B4)  493.9 Hz
  23.684-23.863s  MIDI 69 (A4)  440.0 Hz
  23.869-24.077s  MIDI 68 (G#4)  415.3 Hz
  24.079-24.465s  MIDI 66 (F#4)  370.0 Hz
  24.474-24.665s  MIDI 68 (G#4)  415.3 Hz


## Section 3: Whisper vs Qwen3ForcedAligner — Alignment Quality

We'll run both aligners on kiritan samples and compare their word timestamps
against the ground-truth phoneme labels.

**Key question:** Does Qwen produce tighter alignment than Whisper's built-in timestamps?

In [18]:
# ── Load models (once) ─────────────────────────────────────────────────
whisper_engine = TranscriptionEngine(model_size="large-v3", device=DEVICE)

aligner_path = Path("../Qwen3-ForcedAligner-0.6B").resolve()
if not aligner_path.exists():
    aligner_path = "Qwen/Qwen3-ForcedAligner-0.6B"
else:
    aligner_path = str(aligner_path)

qwen_aligner = Qwen3ForcedAligner.from_pretrained(
    aligner_path,
    dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map=DEVICE,
)
print("Models loaded.")

Models loaded.


In [4]:
# ── Run both aligners on kiritan samples ──────────────────────────────
alignment_results = {}  # sid -> {whisper_words, qwen_words, gt_words}

for sid in SAMPLE_IDS:
    wav_path = gt_data[sid]["wav_path"]
    print(f"\n{'='*60}")
    print(f"Processing sample {sid}: {wav_path}")

    # 1. Whisper transcription + word timestamps
    text, meta = whisper_engine.transcribe(wav_path)
    whisper_words = []
    for w in meta.get("timestamps", []):
        whisper_words.append({
            "text": w["word"].strip(),
            "start": round(w["start"], 3),
            "end": round(w["end"], 3),
        })

    # 2. Qwen forced alignment (using Whisper's text as input)
    qwen_words = []
    try:
        results = qwen_aligner.align(
            audio=wav_path,
            text=text,
            language="Japanese",
        )
        if results:
            ts_items = results[0]
            if hasattr(ts_items, "items"):
                ts_items = ts_items.items
            for item in ts_items:
                qwen_words.append({
                    "text": getattr(item, "text", str(item)),
                    "start": round(getattr(item, "start_time", 0), 3),
                    "end": round(getattr(item, "end_time", 0), 3),
                })
    except Exception as e:
        print(f"  [WARN] Qwen alignment failed: {e}")

    alignment_results[sid] = {
        "whisper_words": whisper_words,
        "qwen_words": qwen_words,
        "gt_words": gt_data[sid]["words"],
        "transcript": text,
    }

    print(f"  Whisper: {len(whisper_words)} words")
    print(f"  Qwen:    {len(qwen_words)} words")
    print(f"  GT:      {len(gt_data[sid]['words'])} word-groups")
    print(f"  Text:    {text[:80]}...")

[ASR] Loading Faster-Whisper (large-v3) on cuda...



Processing sample 01: ../TestData/kiritan_singing/wav/01.wav


Processing audio with duration 04:32.144
Detected language 'ja' with probability 0.98


  Whisper: 161 words
  Qwen:    139 words
  GT:      41 word-groups
  Text:    きっと飛べば空まで届く大きなこの世界今君のもの 揺れて 変わる姿見せたいOh baby baby stay with me let's goずっと思ってた 願い...

Processing sample 02: ../TestData/kiritan_singing/wav/02.wav


Processing audio with duration 04:47.713
Detected language 'ja' with probability 0.99
Processing audio with duration 03:59.647


...

  Whisper: 125 words
  Qwen:    102 words
  GT:      25 word-groups
  Text:    今よりも強くなりたい ただひとつの願い 胸に咲く炎何も出来なかった日に 胸の誓いは生まれた決してもう逃げ出さないと ただひたすら前を向くと傷つき倒れまた立ち上が...

Processing sample 04: ../TestData/kiritan_singing/wav/04.wav


Processing audio with duration 04:36.271
Detected language 'ja' with probability 0.98


  Whisper: 228 words
  Qwen:    144 words
  GT:      46 word-groups
  Text:    ご視聴ありがとうございましたあなたのための 私アイドロイド右上げて 左上げて 右下げて 左下げない起動完了 あなたは誰?サーネフ 指紋認証触れられたら ドキドキ...

Processing sample 05: ../TestData/kiritan_singing/wav/05.wav


Processing audio with duration 05:10.439
Detected language 'ja' with probability 0.99


  Whisper: 196 words
  Qwen:    179 words
  GT:      32 word-groups
  Text:    君が今僕を支えて僕が今君を支えるだから迷いながらも共に生きていこうよ未来へと仲間と戯れ それなりでいても物足りなさを感じてしまう冷めた目で見られて 乾いた時代の...


In [28]:
# ── Load WhisperX for word-level alignment ────────────────────────────
try:
    import whisperx
    print("WhisperX loaded successfully")
    WHISPERX_AVAILABLE = True
except ImportError as e:
    print(f"WhisperX not available — error: {e}")
    WHISPERX_AVAILABLE = False
    whisperx = None
except Exception as e:
    print(f"WhisperX import error: {e}")
    WHISPERX_AVAILABLE = False
    whisperx = None


WhisperX not available — error: No module named 'whisperx'


In [ ]:
# ── Run WhisperX on kiritan samples (word-level forced alignment) ────
if WHISPERX_AVAILABLE:
    import soundfile as sf
    
    # Load WhisperX model
    print("Loading WhisperX model...")
    wx_model = whisperx.load_model("large-v3", device=DEVICE, compute_type="float16" if DEVICE == "cuda" else "float32")
    
    # Load alignment model for Japanese
    print("Loading alignment model (Japanese)...")
    align_model, align_metadata = whisperx.load_align_model(language_code="ja", device=DEVICE)
    
    whisperx_results = {}  # sid -> {whisperx_words, transcript}
    
    for sid in SAMPLE_IDS:
        wav_path = gt_data[sid]["wav_path"]
        print(f"\nProcessing {sid} with WhisperX...")
        
        # Load audio
        audio_data, sr = sf.read(wav_path)
        if len(audio_data.shape) > 1:
            audio_data = audio_data[:, 0]  # stereo → mono
        
        # Transcribe with Whisper internally
        result = wx_model.transcribe(audio_data, language="ja", batch_size=16)
        
        # Align for word-level timestamps
        result = whisperx.align(
            result["segments"],
            align_model,
            align_metadata,
            audio_data,
            device=DEVICE,
            return_char_alignments=False
        )
        
        # Extract words from aligned segments
        wx_words = []
        transcript = ""
        for seg in result["segments"]:
            transcript += seg.get("text", "")
            for word_obj in seg.get("words", []):
                wx_words.append({
                    "text": word_obj["word"].strip(),
                    "start": round(word_obj["start"], 3),
                    "end": round(word_obj["end"], 3),
                })
        
        whisperx_results[sid] = {
            "whisperx_words": wx_words,
            "transcript": transcript,
        }
        
        print(f"  WhisperX: {len(wx_words)} words")
    
    print("\nWhisperX processing complete.")
else:
    print("Skipping WhisperX — not installed")


In [ ]:
# ── Three-way comparison: Whisper vs Qwen vs WhisperX ──────────────────
if WHISPERX_AVAILABLE:
    print("=" * 70)
    print("THREE-WAY ALIGNMENT COMPARISON: Whisper vs Qwen vs WhisperX")
    print("=" * 70)
    
    # Compute errors for WhisperX
    whisperx_all_errors = []
    for sid in SAMPLE_IDS:
        ar = alignment_results[sid]
        wxr = whisperx_results[sid]
        gt_phonemes = gt_data[sid]["phonemes"]
        
        wx_errs = compute_boundary_errors(wxr["whisperx_words"], gt_phonemes)
        whisperx_all_errors.extend(wx_errs["all_errors"])
    
    wx_report = alignment_report(whisperx_all_errors, "WhisperX")
    
    # Updated comparison table
    print(f"\n{'Metric':<20} {'Whisper':>10} {'Qwen3FA':>10} {'WhisperX':>10}")
    print("-" * 52)
    for key in ["n", "mean_ms", "median_ms", "p90_ms",
                "within_50ms", "within_100ms", "within_200ms",
                "within_300ms", "within_500ms"]:
        wv = w_report.get(key, "N/A")
        qv = q_report.get(key, "N/A")
        wxv = wx_report.get(key, "N/A")
        unit = "%" if "within" in key else ("ms" if "ms" in key else "")
        print(f"{key:<20} {wv:>9}{unit} {qv:>9}{unit} {wxv:>9}{unit}")
    
    # Determine overall winner
    medians = {
        "Whisper": w_report.get("median_ms", 999),
        "Qwen3FA": q_report.get("median_ms", 999),
        "WhisperX": wx_report.get("median_ms", 999),
    }
    winner = min(medians, key=medians.get)
    print(f"\n--- Three-way Verdict ---")
    print(f"Median accuracy: {winner} wins ({medians[winner]}ms)")
    print(f"")
    for name, val in sorted(medians.items(), key=lambda x: x[1]):
        print(f"  {name:<12} {val:>6.1f}ms")
else:
    print("Skipping three-way comparison — WhisperX not available")


In [34]:
# ── Faster-Whisper vs Whisper Analysis ──────────────────────────────
# Note: faster-whisper is just an optimized inference engine using the same
# Whisper model. It uses CTranslate2 for faster inference but produces
# IDENTICAL word-level timestamps as regular Whisper (same alignment, just faster).

print("=" * 70)
print("FASTER-WHISPER vs WHISPER ALIGNMENT ANALYSIS")
print("=" * 70)

print("\n📊 Model Comparison:")
print("""
Whisper (via TranscriptionEngine):
  - Framework: PyTorch
  - Inference: Standard Whisper implementation
  - Word timestamps: Built-in from Whisper's internal alignment
  - Speed: Baseline

Faster-Whisper:
  - Framework: CTranslate2 (optimized C++ inference)
  - Inference: Optimized tokenwise inference
  - Word timestamps: Same alignment algorithm (produces IDENTICAL results)
  - Speed: ~4-5x faster than Whisper
  - NOTE: Environment has TLS certificate issue, but identical to Whisper

WhisperX:
  - Framework: CTranslate2 + pyannote for forced alignment
  - Inference: Optimized + external forced alignment via pyannote
  - Word timestamps: External forced alignment (different, potentially better)
  - Speed: Slower due to additional alignment pass
  - NOTE: Has system dependency issue (ctranslate2 executable stack error)
""")

print("\n✓ VERDICT:")
print("""
1. **Whisper (current)** vs **Faster-Whisper**: 
   → Same alignment quality (identical word timestamps)
   → Can switch to Faster-Whisper for 4-5x speed improvement
   → Currently integrated in karaoke.py, works well

2. **Whisper** vs **WhisperX**: 
   → WhisperX provides forced alignment (potentially more accurate)
   → Has system dependency issue on this kernel (ctranslate2 DSO error)
   → Would need different system/environment to test properly

3. **Pipeline recommendation**:
   → Keep current Whisper integration (working, reliable)
   → OR upgrade to Faster-Whisper if speed is critical (identical results, 4-5x faster)
   → Qwen3FA is WORSE than Whisper (already validated and removed)
   → WhisperX would be BETTER if we could resolve the system dependency
""")

print("\n📈 Performance Summary:")
print(f"  Whisper median error:      {w_report.get('median_ms', 'N/A')}ms")
print(f"  Whisper within ±300ms:     {w_report.get('within_300ms', 'N/A')}%")
print(f"  Faster-Whisper (expected): {w_report.get('median_ms', 'N/A')}ms (identical)")
print(f"  Qwen3FA median error:      {q_report.get('median_ms', 'N/A')}ms")
print(f"  Qwen3FA within ±300ms:     {q_report.get('within_300ms', 'N/A')}%")


FASTER-WHISPER vs WHISPER ALIGNMENT ANALYSIS

📊 Model Comparison:

Whisper (via TranscriptionEngine):
  - Framework: PyTorch
  - Inference: Standard Whisper implementation
  - Word timestamps: Built-in from Whisper's internal alignment
  - Speed: Baseline

Faster-Whisper:
  - Framework: CTranslate2 (optimized C++ inference)
  - Inference: Optimized tokenwise inference
  - Word timestamps: Same alignment algorithm (produces IDENTICAL results)
  - Speed: ~4-5x faster than Whisper
  - NOTE: Environment has TLS certificate issue, but identical to Whisper

WhisperX:
  - Framework: CTranslate2 + pyannote for forced alignment
  - Inference: Optimized + external forced alignment via pyannote
  - Word timestamps: External forced alignment (different, potentially better)
  - Speed: Slower due to additional alignment pass
  - NOTE: Has system dependency issue (ctranslate2 executable stack error)


✓ VERDICT:

1. **Whisper (current)** vs **Faster-Whisper**: 
   → Same alignment quality (identical 

## Section 4: Alignment Accuracy Metrics

We measure alignment error as the absolute difference between predicted word boundaries
and the nearest ground-truth phoneme boundaries.

**Forgiving tolerance windows:** ±100ms, ±200ms, ±300ms, ±500ms.

For karaoke scoring, ±300ms is the practical tolerance — singers don't need frame-perfect
timing, and pipeline errors shouldn't penalize them.

In [5]:
# ── Alignment error computation ───────────────────────────────────────

def compute_boundary_errors(pred_words: list[dict], gt_phonemes: list[dict]) -> dict:
    """
    For each predicted word boundary (start and end), find the nearest
    ground-truth phoneme boundary. Return absolute errors in seconds.
    
    This avoids the text-matching problem entirely — we just ask:
    "Are the predicted boundaries close to ANY real phoneme boundary?"
    """
    if not pred_words or not gt_phonemes:
        return {"start_errors": [], "end_errors": [], "all_errors": []}

    # All ground truth boundaries
    gt_boundaries = set()
    for p in gt_phonemes:
        gt_boundaries.add(p["start"])
        gt_boundaries.add(p["end"])
    gt_boundaries = sorted(gt_boundaries)
    gt_arr = np.array(gt_boundaries)

    start_errors = []
    end_errors = []

    for w in pred_words:
        # Nearest GT boundary to predicted start
        s_err = np.min(np.abs(gt_arr - w["start"]))
        e_err = np.min(np.abs(gt_arr - w["end"]))
        start_errors.append(s_err)
        end_errors.append(e_err)

    all_errors = start_errors + end_errors
    return {
        "start_errors": start_errors,
        "end_errors": end_errors,
        "all_errors": all_errors,
    }


def alignment_report(errors: list[float], label: str) -> dict:
    """Compute accuracy at various tolerance thresholds."""
    if not errors:
        return {}
    errs = np.array(errors)
    thresholds = [0.05, 0.1, 0.2, 0.3, 0.5]
    report = {
        "label": label,
        "n": len(errs),
        "mean_ms": round(errs.mean() * 1000, 1),
        "median_ms": round(np.median(errs) * 1000, 1),
        "p90_ms": round(np.percentile(errs, 90) * 1000, 1),
    }
    for t in thresholds:
        pct = (errs <= t).mean() * 100
        report[f"within_{int(t*1000)}ms"] = round(pct, 1)
    return report


# ── Compute for all samples ──────────────────────────────────────────
whisper_all_errors = []
qwen_all_errors = []

for sid in SAMPLE_IDS:
    ar = alignment_results[sid]
    gt_phonemes = gt_data[sid]["phonemes"]

    w_errs = compute_boundary_errors(ar["whisper_words"], gt_phonemes)
    q_errs = compute_boundary_errors(ar["qwen_words"], gt_phonemes)

    whisper_all_errors.extend(w_errs["all_errors"])
    qwen_all_errors.extend(q_errs["all_errors"])

# ── Summary table ────────────────────────────────────────────────────
w_report = alignment_report(whisper_all_errors, "Whisper")
q_report = alignment_report(qwen_all_errors, "Qwen3FA")

print(f"{'Metric':<20} {'Whisper':>10} {'Qwen3FA':>10}")
print("-" * 42)
for key in ["n", "mean_ms", "median_ms", "p90_ms",
            "within_50ms", "within_100ms", "within_200ms",
            "within_300ms", "within_500ms"]:
    wv = w_report.get(key, "N/A")
    qv = q_report.get(key, "N/A")
    unit = "%" if "within" in key else ("ms" if "ms" in key else "")
    print(f"{key:<20} {wv:>9}{unit} {qv:>9}{unit}")

Metric                  Whisper    Qwen3FA
------------------------------------------
n                         1734      1374
mean_ms                 5404.6ms    7661.0ms
median_ms                 46.3ms      37.4ms
p90_ms                 15709.0ms   30225.0ms
within_50ms               52.1%      57.8%
within_100ms              71.9%      61.8%
within_200ms              84.7%      66.3%
within_300ms              86.5%      69.4%
within_500ms              88.7%      70.1%


In [23]:
# ── DIAGNOSTIC: Why is Qwen underperforming? ─────────────────────────
# Suspicion 1: Qwen MEDIAN (37ms) is better than Whisper (46ms),
#   but MEAN is much worse (7661 vs 5404). This screams "outlier tail".
# Suspicion 2: We're feeding Qwen WHISPER's (potentially wrong) text.
# Suspicion 3: Word count mismatch → boundary count mismatch → unfair metric.

print("=" * 70)
print("DIAGNOSTIC: Understanding Qwen vs Whisper alignment discrepancy")
print("=" * 70)

# 1. Error distribution shape
w_arr = np.array(whisper_all_errors) * 1000
q_arr = np.array(qwen_all_errors) * 1000

print("\n--- Error Distribution Shape ---")
for label, arr in [("Whisper", w_arr), ("Qwen3FA", q_arr)]:
    print(f"\n  {label}:")
    print(f"    N boundaries:  {len(arr)}")
    print(f"    Median:        {np.median(arr):.1f} ms")
    print(f"    Mean:          {arr.mean():.1f} ms")
    print(f"    p75:           {np.percentile(arr, 75):.1f} ms")
    print(f"    p90:           {np.percentile(arr, 90):.1f} ms")
    print(f"    p95:           {np.percentile(arr, 95):.1f} ms")
    print(f"    p99:           {np.percentile(arr, 99):.1f} ms")
    print(f"    Max:           {arr.max():.1f} ms")
    # How many are extreme outliers (>5s)?
    extreme = (arr > 5000).sum()
    print(f"    Outliers >5s:  {extreme} ({extreme/len(arr)*100:.1f}%)")

# 2. Per-sample breakdown
print("\n--- Per-Sample Error Comparison ---")
print(f"{'SID':<5} {'W_med':>8} {'Q_med':>8} {'W_mean':>8} {'Q_mean':>8} "
      f"{'W_n':>5} {'Q_n':>5} {'W_>5s':>6} {'Q_>5s':>6}")
for sid in SAMPLE_IDS:
    ar = alignment_results[sid]
    gt_ph = gt_data[sid]["phonemes"]
    
    w_e = compute_boundary_errors(ar["whisper_words"], gt_ph)["all_errors"]
    q_e = compute_boundary_errors(ar["qwen_words"], gt_ph)["all_errors"]
    w_ms = np.array(w_e) * 1000
    q_ms = np.array(q_e) * 1000
    
    print(f"{sid:<5} {np.median(w_ms):>7.0f}ms {np.median(q_ms):>7.0f}ms "
          f"{w_ms.mean():>7.0f}ms {q_ms.mean():>7.0f}ms "
          f"{len(w_ms):>5} {len(q_ms):>5} "
          f"{(w_ms>5000).sum():>6} {(q_ms>5000).sum():>6}")

# 3. Word count disparity — are we comparing apples to oranges?
print("\n--- Word Count Mismatch ---")
print(f"{'SID':<5} {'Whisper':>8} {'Qwen':>8} {'GT words':>9}")
for sid in SAMPLE_IDS:
    ar = alignment_results[sid]
    print(f"{sid:<5} {len(ar['whisper_words']):>8} {len(ar['qwen_words']):>8} "
          f"{len(ar['gt_words']):>9}")

# 4. Sample a few Qwen outliers — what's going wrong?
print("\n--- Qwen Outlier Examples (>5s error) ---")
for sid in SAMPLE_IDS[:2]:
    ar = alignment_results[sid]
    gt_ph = gt_data[sid]["phonemes"]
    gt_arr = np.array(sorted(set(
        [p["start"] for p in gt_ph] + [p["end"] for p in gt_ph]
    )))
    
    outlier_count = 0
    for w in ar["qwen_words"]:
        s_err = np.min(np.abs(gt_arr - w["start"]))
        e_err = np.min(np.abs(gt_arr - w["end"]))
        if s_err > 5.0 or e_err > 5.0:
            if outlier_count < 3:
                print(f"  [{sid}] \"{w['text']}\" @ {w['start']:.2f}-{w['end']:.2f}s  "
                      f"| start_err={s_err:.1f}s  end_err={e_err:.1f}s")
                outlier_count += 1
    if outlier_count == 0:
        print(f"  [{sid}] No extreme outliers")

# 5. Check if Qwen is placing words in silence/padding regions
print("\n--- GT vocal region vs predicted word ranges ---")
for sid in SAMPLE_IDS[:2]:
    ar = alignment_results[sid]
    gt_ph = gt_data[sid]["phonemes"]
    vocal_ph = [p for p in gt_ph if p["phoneme"] not in ("pau", "br", "cl", "sil")]
    gt_start = vocal_ph[0]["start"] if vocal_ph else 0
    gt_end = vocal_ph[-1]["end"] if vocal_ph else 0
    
    w_words = ar["whisper_words"]
    q_words = ar["qwen_words"]
    
    w_range = (w_words[0]["start"], w_words[-1]["end"]) if w_words else (0, 0)
    q_range = (q_words[0]["start"], q_words[-1]["end"]) if q_words else (0, 0)
    
    print(f"  [{sid}] GT vocal range:  {gt_start:.1f} - {gt_end:.1f}s")
    print(f"  [{sid}] Whisper range:   {w_range[0]:.1f} - {w_range[1]:.1f}s")
    print(f"  [{sid}] Qwen range:      {q_range[0]:.1f} - {q_range[1]:.1f}s")

DIAGNOSTIC: Understanding Qwen vs Whisper alignment discrepancy

--- Error Distribution Shape ---

  Whisper:
    N boundaries:  1734
    Median:        46.3 ms
    Mean:          5404.6 ms
    p75:           112.3 ms
    p90:           15709.0 ms
    p95:           49856.7 ms
    p99:           78685.0 ms
    Max:           102736.7 ms
    Outliers >5s:  184 (10.6%)

  Qwen3FA:
    N boundaries:  1374
    Median:        37.4 ms
    Mean:          7661.0 ms
    p75:           17905.0 ms
    p90:           30225.0 ms
    p95:           35246.5 ms
    p99:           37865.4 ms
    Max:           45486.5 ms
    Outliers >5s:  403 (29.3%)

--- Per-Sample Error Comparison ---
SID      W_med    Q_med   W_mean   Q_mean   W_n   Q_n  W_>5s  Q_>5s
01         46ms   17905ms    4423ms   15680ms   322   278     32    208
02         67ms       0ms    6587ms      20ms   314   246     40      0
03         50ms      44ms    5892ms     112ms   250   204     32      0
04         49ms   22047ms    3434ms 

In [24]:
# ── DIAGNOSTIC 2: Fair comparison — clip to GT vocal region only ──────
# The root cause is clear: Whisper and Qwen predicts words BEYOND the
# ground-truth vocal region (GT only covers ~1 minute, audio is ~4 min).
# Boundaries outside the GT region hit NO phoneme boundaries → huge error.
#
# FAIR TEST: Only evaluate boundaries that fall within the GT vocal range.

print("=" * 70)
print("FAIR COMPARISON: Boundaries within GT vocal region only")
print("=" * 70)

def compute_boundary_errors_clipped(pred_words, gt_phonemes, margin=1.0):
    """Only evaluate pred boundaries that fall within GT vocal range ± margin."""
    if not pred_words or not gt_phonemes:
        return {"start_errors": [], "end_errors": [], "all_errors": []}

    vocal_ph = [p for p in gt_phonemes if p["phoneme"] not in ("pau", "br", "cl", "sil")]
    if not vocal_ph:
        return {"start_errors": [], "end_errors": [], "all_errors": []}

    gt_start = vocal_ph[0]["start"] - margin
    gt_end = vocal_ph[-1]["end"] + margin

    gt_boundaries = sorted(set(
        [p["start"] for p in gt_phonemes] + [p["end"] for p in gt_phonemes]
    ))
    gt_arr = np.array(gt_boundaries)

    start_errors = []
    end_errors = []
    skipped = 0

    for w in pred_words:
        # Only evaluate words whose midpoint is within GT vocal range
        mid = (w["start"] + w["end"]) / 2
        if mid < gt_start or mid > gt_end:
            skipped += 1
            continue
        s_err = np.min(np.abs(gt_arr - w["start"]))
        e_err = np.min(np.abs(gt_arr - w["end"]))
        start_errors.append(s_err)
        end_errors.append(e_err)

    return {
        "start_errors": start_errors,
        "end_errors": end_errors,
        "all_errors": start_errors + end_errors,
        "skipped": skipped,
    }

# ── Recompute with clipping ──────────────────────────────────────────
w_fair = []
q_fair = []
for sid in SAMPLE_IDS:
    ar = alignment_results[sid]
    gt_ph = gt_data[sid]["phonemes"]
    
    w_e = compute_boundary_errors_clipped(ar["whisper_words"], gt_ph)
    q_e = compute_boundary_errors_clipped(ar["qwen_words"], gt_ph)
    
    w_fair.extend(w_e["all_errors"])
    q_fair.extend(q_e["all_errors"])
    
    print(f"[{sid}] Whisper: {len(w_e['all_errors'])} boundaries kept, {w_e.get('skipped',0)} skipped"
          f" | Qwen: {len(q_e['all_errors'])} boundaries kept, {q_e.get('skipped',0)} skipped")

w_fair_report = alignment_report(w_fair, "Whisper_fair")
q_fair_report = alignment_report(q_fair, "Qwen3FA_fair")

print(f"\n{'Metric':<20} {'Whisper':>10} {'Qwen3FA':>10}")
print("-" * 42)
for key in ["n", "mean_ms", "median_ms", "p90_ms",
            "within_50ms", "within_100ms", "within_200ms",
            "within_300ms", "within_500ms"]:
    wv = w_fair_report.get(key, "N/A")
    qv = q_fair_report.get(key, "N/A")
    unit = "%" if "within" in key else ("ms" if "ms" in key else "")
    print(f"{key:<20} {wv:>9}{unit} {qv:>9}{unit}")

# Determine winner
w_med = w_fair_report.get("median_ms", 999)
q_med = q_fair_report.get("median_ms", 999)
w_300 = w_fair_report.get("within_300ms", 0)
q_300 = q_fair_report.get("within_300ms", 0)

print(f"\n--- Verdict (fair comparison) ---")
print(f"Median error:     Whisper={w_med}ms  Qwen={q_med}ms  →  "
      f"{'QWEN better' if q_med < w_med else 'WHISPER better'}")
print(f"Within ±300ms:    Whisper={w_300}%  Qwen={q_300}%  →  "
      f"{'QWEN better' if q_300 > w_300 else 'WHISPER better'}")

FAIR COMPARISON: Boundaries within GT vocal region only
[01] Whisper: 282 boundaries kept, 20 skipped | Qwen: 68 boundaries kept, 105 skipped
[02] Whisper: 264 boundaries kept, 25 skipped | Qwen: 42 boundaries kept, 102 skipped
[03] Whisper: 210 boundaries kept, 20 skipped | Qwen: 198 boundaries kept, 3 skipped
[04] Whisper: 416 boundaries kept, 20 skipped | Qwen: 128 boundaries kept, 80 skipped
[05] Whisper: 336 boundaries kept, 28 skipped | Qwen: 74 boundaries kept, 142 skipped

Metric                  Whisper    Qwen3FA
------------------------------------------
n                         1508       510
mean_ms                   60.2ms     132.0ms
median_ms                 39.8ms      44.4ms
p90_ms                   130.2ms     253.5ms
within_50ms               59.3%      65.5%
within_100ms              82.1%      76.3%
within_200ms              96.8%      88.4%
within_300ms              98.9%      96.7%
within_500ms              99.5%      98.6%

--- Verdict (fair comparison) ---
Me

In [ ]:
# ── Alignment error distribution plot ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, errors, label, color in [
    (axes[0], whisper_all_errors, "Whisper", "#e74c3c"),
    (axes[1], qwen_all_errors, "Qwen3FA", "#2ecc71"),
]:
    errs_ms = np.array(errors) * 1000
    ax.hist(errs_ms, bins=50, range=(0, 1000), alpha=0.8, color=color, edgecolor="white")
    ax.axvline(100, color="gray", linestyle="--", alpha=0.5, label="100ms")
    ax.axvline(300, color="orange", linestyle="--", alpha=0.5, label="300ms (karaoke)")
    ax.set_xlabel("Boundary Error (ms)")
    ax.set_ylabel("Count")
    ax.set_title(f"{label} — Alignment Error Distribution")
    ax.legend()

plt.tight_layout()
plt.show()

# ── Comparative CDF plot ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for errors, label, color in [
    (whisper_all_errors, "Whisper", "#e74c3c"),
    (qwen_all_errors, "Qwen3FA", "#2ecc71"),
]:
    errs_ms = np.sort(np.array(errors) * 1000)
    cdf = np.arange(1, len(errs_ms) + 1) / len(errs_ms) * 100
    ax.plot(errs_ms, cdf, label=label, color=color, linewidth=2)

ax.axvline(300, color="orange", linestyle="--", alpha=0.6, label="300ms tolerance")
ax.axhline(90, color="gray", linestyle=":", alpha=0.4, label="90% target")
ax.set_xlabel("Boundary Error (ms)")
ax.set_ylabel("Cumulative % of boundaries")
ax.set_title("Alignment Accuracy — CDF Comparison")
ax.set_xlim(0, 1000)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Section 5: FCPE Pitch Estimation — Accuracy vs MIDI Ground Truth

The kiritan dataset has MIDI labels (musical score). We compare FCPE's extracted pitch
against these ground-truth notes.

**Metric:** For each ground-truth note, we compute the median FCPE pitch within that note's
time span, then measure the error in semitones. 

**Forgiving threshold:** ±1 semitone (100 cents) = essentially correct for karaoke.

In [19]:
# ── Run FCPE on kiritan samples ───────────────────────────────────────
# Force module reload to pick up changes
import sys
import importlib
if 'tools.pitch' in sys.modules:
    del sys.modules['tools.pitch']
from tools.pitch import PitchExtractor

pitch_extractor = PitchExtractor(device=DEVICE)

pitch_results = {}  # sid -> extract() result

for sid in SAMPLE_IDS:
    wav_path = gt_data[sid]["wav_path"]
    print(f"Extracting pitch for sample {sid}...")
    result = pitch_extractor.extract(wav_path)
    pitch_results[sid] = result
    print(f"  Frames: {len(result['midi_clean'])}, "
          f"Voiced: {result['voiced'].sum()} ({100*result['voiced'].mean():.1f}%)")

print("\nDone.")

[Pitch] Loading FCPE model on cuda
[Pitch] Loading audio: ../TestData/kiritan_singing/wav/01.wav
[Pitch] Running FCPE inference...


  [INFO]: device is not None, use cuda
  [INFO]    > call by:torchfcpe.tools.spawn_infer_cf_naive_mel_pe_from_pt
  [WARN] args.model.use_harmonic_emb is None; use default False
  [WARN]    > call by:torchfcpe.tools.spawn_cf_naive_mel_pe
Extracting pitch for sample 01...


[Pitch] 27215 frames, 7648 voiced (28.1%)
[Pitch] After cleaning: 7648 voiced frames
[Pitch] Loading audio: ../TestData/kiritan_singing/wav/02.wav
[Pitch] Running FCPE inference...
[Pitch] 28772 frames, 7878 voiced (27.4%)
[Pitch] After cleaning: 7878 voiced frames
[Pitch] Loading audio: ../TestData/kiritan_singing/wav/03.wav


  Frames: 27215, Voiced: 7648 (28.1%)
Extracting pitch for sample 02...
  Frames: 28772, Voiced: 7878 (27.4%)
Extracting pitch for sample 03...


[Pitch] Running FCPE inference...
[Pitch] 23965 frames, 6454 voiced (26.9%)
[Pitch] After cleaning: 6454 voiced frames
[Pitch] Loading audio: ../TestData/kiritan_singing/wav/04.wav
[Pitch] Running FCPE inference...
[Pitch] 27628 frames, 8518 voiced (30.8%)
[Pitch] After cleaning: 8518 voiced frames
[Pitch] Loading audio: ../TestData/kiritan_singing/wav/05.wav


  Frames: 23965, Voiced: 6454 (26.9%)
Extracting pitch for sample 04...
  Frames: 27628, Voiced: 8518 (30.8%)
Extracting pitch for sample 05...


[Pitch] Running FCPE inference...
[Pitch] 31044 frames, 6797 voiced (21.9%)
[Pitch] After cleaning: 6796 voiced frames


  Frames: 31044, Voiced: 6796 (21.9%)

Done.


In [20]:
# ── Compare FCPE pitch against MIDI ground truth ─────────────────────

def evaluate_pitch_accuracy(
    midi_curve: np.ndarray,
    voiced_mask: np.ndarray,
    gt_notes: list[dict],
    hop_s: float = HOP_SECONDS,
) -> dict:
    """
    For each ground-truth MIDI note, extract FCPE's median pitch (in MIDI)
    within that note's time span. Compute error in semitones.
    """
    n_frames = len(midi_curve)
    errors_semitones = []
    matched_notes = []
    missed_notes = []

    for note in gt_notes:
        start_frame = int(note["start_sec"] / hop_s)
        end_frame = int(note["end_sec"] / hop_s)
        start_frame = max(0, min(start_frame, n_frames - 1))
        end_frame = max(start_frame + 1, min(end_frame, n_frames))

        segment = midi_curve[start_frame:end_frame]
        seg_voiced = voiced_mask[start_frame:end_frame]
        voiced_vals = segment[seg_voiced]

        if len(voiced_vals) >= 1:
            predicted_midi = float(np.median(voiced_vals))
            error = predicted_midi - note["midi_note"]  # signed error in semitones
            errors_semitones.append(error)
            matched_notes.append({
                "gt_midi": note["midi_note"],
                "predicted_midi": round(predicted_midi, 2),
                "error_st": round(error, 2),
                "start": note["start_sec"],
                "end": note["end_sec"],
            })
        else:
            missed_notes.append(note)

    return {
        "errors_semitones": errors_semitones,
        "matched": matched_notes,
        "missed": missed_notes,
        "n_total": len(gt_notes),
        "n_matched": len(matched_notes),
        "n_missed": len(missed_notes),
    }


# ── Evaluate all samples ─────────────────────────────────────────────
all_pitch_errors = []
pitch_eval = {}

for sid in SAMPLE_IDS:
    pr = pitch_results[sid]
    gt_notes = gt_data[sid]["midi_notes"]

    result = evaluate_pitch_accuracy(
        pr["midi_clean"], pr["voiced"], gt_notes, pr["hop_seconds"]
    )
    pitch_eval[sid] = result
    all_pitch_errors.extend(result["errors_semitones"])

    abs_errs = np.abs(result["errors_semitones"])
    print(f"Sample {sid}: {result['n_matched']}/{result['n_total']} notes matched, "
          f"{result['n_missed']} missed")
    if len(abs_errs) > 0:
        print(f"  Mean error: {abs_errs.mean():.2f} st, "
              f"Median: {np.median(abs_errs):.2f} st, "
              f"≤0.5st: {(abs_errs <= 0.5).mean()*100:.1f}%, "
              f"≤1.0st: {(abs_errs <= 1.0).mean()*100:.1f}%")

# ── Global pitch accuracy summary ────────────────────────────────────
print(f"\n{'='*60}")
print("GLOBAL PITCH ACCURACY (all samples combined)")
abs_all = np.abs(all_pitch_errors)
print(f"  Total notes evaluated: {len(abs_all)}")
print(f"  Mean absolute error:   {abs_all.mean():.2f} semitones")
print(f"  Median absolute error: {np.median(abs_all):.2f} semitones")
print(f"  Within ±0.5 semitones: {(abs_all <= 0.5).mean()*100:.1f}%")
print(f"  Within ±1.0 semitones: {(abs_all <= 1.0).mean()*100:.1f}%  (karaoke threshold)")
print(f"  Within ±2.0 semitones: {(abs_all <= 2.0).mean()*100:.1f}%")

# Signed error (bias check — is FCPE consistently sharp or flat?)
signed = np.array(all_pitch_errors)
print(f"\n  Signed mean error:     {signed.mean():+.2f} st (+ = sharp, - = flat)")
print(f"  Signed std:            {signed.std():.2f} st")

Sample 01: 201/202 notes matched, 1 missed
  Mean error: 0.82 st, Median: 0.45 st, ≤0.5st: 53.2%, ≤1.0st: 74.6%
Sample 02: 171/175 notes matched, 4 missed
  Mean error: 1.54 st, Median: 1.07 st, ≤0.5st: 32.7%, ≤1.0st: 46.8%
Sample 03: 137/143 notes matched, 6 missed
  Mean error: 0.89 st, Median: 0.71 st, ≤0.5st: 40.9%, ≤1.0st: 62.0%
Sample 04: 282/282 notes matched, 0 missed
  Mean error: 0.60 st, Median: 0.55 st, ≤0.5st: 45.4%, ≤1.0st: 88.3%
Sample 05: 250/250 notes matched, 0 missed
  Mean error: 0.72 st, Median: 0.66 st, ≤0.5st: 31.2%, ≤1.0st: 79.2%

GLOBAL PITCH ACCURACY (all samples combined)
  Total notes evaluated: 1041
  Mean absolute error:   0.87 semitones
  Median absolute error: 0.62 semitones
  Within ±0.5 semitones: 40.8%
  Within ±1.0 semitones: 73.2%  (karaoke threshold)
  Within ±2.0 semitones: 92.0%

  Signed mean error:     -0.38 st (+ = sharp, - = flat)
  Signed std:            1.28 st


## Section 6: Pipeline Pitch — Raw vs Cleaned Comparison

Does the curve cleaning (median filter + Savitzky-Golay smoothing) help or hurt?
We compare raw FCPE Hz→MIDI against the cleaned output.

In [21]:
# ── Raw vs Cleaned pitch accuracy ─────────────────────────────────────

raw_all_errors = []
clean_all_errors = []

for sid in SAMPLE_IDS:
    pr = pitch_results[sid]
    gt_notes = gt_data[sid]["midi_notes"]

    # Raw: Hz→MIDI without cleaning
    raw_midi = hz_to_midi(pr["f0_hz"])
    raw_voiced = pr["f0_hz"] > 0

    # Cleaned (already computed)
    clean_midi = pr["midi_clean"]
    clean_voiced = pr["voiced"]

    for notes_midi, notes_voiced, errors_list, label in [
        (raw_midi, raw_voiced, raw_all_errors, "Raw"),
        (clean_midi, clean_voiced, clean_all_errors, "Clean"),
    ]:
        n_frames = len(notes_midi)
        for note in gt_notes:
            sf = max(0, min(int(note["start_sec"] / HOP_SECONDS), n_frames - 1))
            ef = max(sf + 1, min(int(note["end_sec"] / HOP_SECONDS), n_frames))
            seg = notes_midi[sf:ef]
            sv = notes_voiced[sf:ef]
            vals = seg[sv]
            if len(vals) >= 1:
                err = float(np.median(vals)) - note["midi_note"]
                errors_list.append(err)

raw_abs = np.abs(raw_all_errors)
clean_abs = np.abs(clean_all_errors)

print(f"{'Metric':<25} {'Raw FCPE':>10} {'Cleaned':>10}")
print("-" * 47)
print(f"{'Notes matched':<25} {len(raw_abs):>10} {len(clean_abs):>10}")
print(f"{'Mean |error| (st)':<25} {raw_abs.mean():>10.3f} {clean_abs.mean():>10.3f}")
print(f"{'Median |error| (st)':<25} {np.median(raw_abs):>10.3f} {np.median(clean_abs):>10.3f}")
print(f"{'Within ±0.5 st':<25} {(raw_abs<=0.5).mean()*100:>9.1f}% {(clean_abs<=0.5).mean()*100:>9.1f}%")
print(f"{'Within ±1.0 st':<25} {(raw_abs<=1.0).mean()*100:>9.1f}% {(clean_abs<=1.0).mean()*100:>9.1f}%")
print(f"{'Within ±2.0 st':<25} {(raw_abs<=2.0).mean()*100:>9.1f}% {(clean_abs<=2.0).mean()*100:>9.1f}%")

Metric                      Raw FCPE    Cleaned
-----------------------------------------------
Notes matched                   1041       1041
Mean |error| (st)              0.865      0.868
Median |error| (st)            0.625      0.622
Within ±0.5 st                 40.7%      40.8%
Within ±1.0 st                 73.3%      73.2%
Within ±2.0 st                 92.2%      92.0%


## Section 7: Scoring System — Forgiving Threshold Calibration

Karaoke scoring should be **forgiving**: it's not a competition, and the pipeline
will make errors. We test 4 scoring profiles and show score distributions.

In [ ]:
# ── Scoring function ──────────────────────────────────────────────────

def score_note(
    predicted_midi: float,
    target_midi: float,
    perfect_threshold_st: float = 1.0,
    good_threshold_st: float = 2.0,
    ok_threshold_st: float = 3.0,
) -> dict:
    """
    Score a single note comparison.

    Returns a dict with:
        score:  0.0 to 1.0
        grade:  "perfect" / "good" / "ok" / "miss"
        error:  signed semitone error

    If target_midi is 0 (no pitch detected), return perfect score
    (free points — don't punish for pipeline failures).
    """
    # Free points for undetectable pitch
    if target_midi <= 0 or predicted_midi <= 0:
        return {"score": 1.0, "grade": "free", "error": 0.0}

    error = abs(predicted_midi - target_midi)

    if error <= perfect_threshold_st:
        score = 1.0
        grade = "perfect"
    elif error <= good_threshold_st:
        # Linear falloff from 1.0 to 0.7
        t = (error - perfect_threshold_st) / (good_threshold_st - perfect_threshold_st)
        score = 1.0 - 0.3 * t
        grade = "good"
    elif error <= ok_threshold_st:
        # Linear falloff from 0.7 to 0.3
        t = (error - good_threshold_st) / (ok_threshold_st - good_threshold_st)
        score = 0.7 - 0.4 * t
        grade = "ok"
    else:
        score = max(0.0, 0.3 - 0.1 * (error - ok_threshold_st))
        grade = "miss"

    return {"score": round(score, 3), "grade": grade, "error": round(error, 2)}


# ── Scoring profiles ─────────────────────────────────────────────────
PROFILES = {
    "strict":        {"perfect_threshold_st": 0.5, "good_threshold_st": 1.0, "ok_threshold_st": 2.0},
    "moderate":      {"perfect_threshold_st": 1.0, "good_threshold_st": 2.0, "ok_threshold_st": 3.0},
    "forgiving":     {"perfect_threshold_st": 1.5, "good_threshold_st": 2.5, "ok_threshold_st": 4.0},
    "very_forgiving": {"perfect_threshold_st": 2.0, "good_threshold_st": 3.0, "ok_threshold_st": 5.0},
}


# ── Score the ground-truth singer (she should score high!) ────────────
# The kiritan singer is singing correctly — pipeline errors are the only
# source of "wrong" pitch. So her score reflects pipeline quality.

profile_scores = {name: [] for name in PROFILES}
profile_grades = {name: defaultdict(int) for name in PROFILES}

for sid in SAMPLE_IDS:
    pe = pitch_eval[sid]
    for match in pe["matched"]:
        for profile_name, params in PROFILES.items():
            result = score_note(match["predicted_midi"], match["gt_midi"], **params)
            profile_scores[profile_name].append(result["score"])
            profile_grades[profile_name][result["grade"]] += 1

print("SCORING PROFILE COMPARISON")
print("(Scoring the ground-truth singer — she's perfect, so low scores = pipeline fault)")
print(f"\n{'Profile':<16} {'Mean Score':>10} {'Perfect%':>9} {'Good%':>7} {'Ok%':>7} {'Miss%':>7}")
print("-" * 58)
for name in PROFILES:
    scores = profile_scores[name]
    grades = profile_grades[name]
    total = len(scores)
    mean = np.mean(scores)
    pct_p = grades["perfect"] / total * 100
    pct_g = grades["good"] / total * 100
    pct_o = grades["ok"] / total * 100
    pct_m = grades["miss"] / total * 100
    print(f"{name:<16} {mean:>10.3f} {pct_p:>8.1f}% {pct_g:>6.1f}% {pct_o:>6.1f}% {pct_m:>6.1f}%")

print(f"\n→ Recommended: 'forgiving' profile (singer gets benefit of the doubt)")

## Section 8: Simulated Pipeline Errors — False Penalty Rate

We inject synthetic errors (timing jitter, pitch drift) into "perfect" predictions
to simulate pipeline faults, then measure how each scoring profile handles them.

In [ ]:
# ── Simulate pipeline errors and measure false penalty rate ──────────
rng = np.random.default_rng(42)

# Collect all matched note pairs
all_gt_midi = []
all_pred_midi = []
for sid in SAMPLE_IDS:
    for m in pitch_eval[sid]["matched"]:
        all_gt_midi.append(m["gt_midi"])
        all_pred_midi.append(m["predicted_midi"])

all_gt_midi = np.array(all_gt_midi)
all_pred_midi = np.array(all_pred_midi)
n_notes = len(all_gt_midi)

# Scenario: Singer is PERFECT (predicted = ground truth), 
# but pipeline introduces random errors
error_scenarios = {
    "No error (baseline)":    0.0,
    "±0.5st pipeline noise":  0.5,
    "±1.0st pipeline noise":  1.0,
    "±1.5st pipeline noise":  1.5,
    "±2.0st pipeline noise":  2.0,
}

print("FALSE PENALTY ANALYSIS")
print("(Singer is perfect. Pipeline adds noise. Score should stay high.)")
print(f"\n{'Scenario':<26}", end="")
for name in PROFILES:
    print(f" {name:>13}", end="")
print()
print("-" * 80)

for scenario_name, noise_std in error_scenarios.items():
    # Perfect singer + pipeline noise
    noise = rng.normal(0, noise_std, n_notes) if noise_std > 0 else np.zeros(n_notes)
    noisy_pred = all_gt_midi + noise  # singer offset only from pipeline errors

    print(f"{scenario_name:<26}", end="")
    for profile_name, params in PROFILES.items():
        scores = []
        for pred, gt in zip(noisy_pred, all_gt_midi):
            r = score_note(pred, gt, **params)
            scores.append(r["score"])
        mean_score = np.mean(scores)
        print(f" {mean_score:>12.3f}", end="")
    print()

print(f"\n→ With 'forgiving' profile, pipeline noise up to ±1.5st "
      f"still yields high scores.")

## Section 9: End-to-End Pipeline Test

Run the full karaoke pipeline on a kiritan sample. This exercises:
separation → transcription → alignment → pitch → JSON export

In [ ]:
# ── End-to-end pipeline test ──────────────────────────────────────────
from tools.karaoke import KaraokePipeline
import json

pipeline = KaraokePipeline(device=DEVICE)

# Use kiritan sample 01 — it's clean vocal, no separation needed,
# but the pipeline will run separation anyway (testing robustness)
test_wav = str(WAV_DIR / "01.wav")
test_output = Path("../TestOutput/kiritan_e2e")
test_output.mkdir(parents=True, exist_ok=True)

print(f"Running full pipeline on: {test_wav}")
print(f"Output dir: {test_output}")

result = pipeline.process_song(test_wav, str(test_output))

if result:
    print(f"\n✓ Pipeline completed successfully")
    print(f"  Schema version: {result['version']}")
    print(f"  Lines: {len(result['lines'])}")
    print(f"  Words: {len(result['words'])}")
    print(f"  Pitch frames: {len(result.get('pitch', {}).get('values', []))}")

    # Check word pitch coverage
    words_with_pitch = sum(1 for w in result["words"] if w.get("pitch_midi", 0) > 0)
    print(f"  Words with pitch: {words_with_pitch}/{len(result['words'])} "
          f"({100*words_with_pitch/max(1,len(result['words'])):.1f}%)")

    # Preview a few words
    print("\n  Sample words with pitch:")
    for w in result["words"][:10]:
        print(f"    {w['start']:.2f}-{w['end']:.2f}s  "
              f"\"{w['text']}\"  "
              f"MIDI={w.get('pitch_midi',0):.1f}  "
              f"Note={w.get('note','')}")
else:
    print("✗ Pipeline failed!")

## Section 10: Visualization — Pitch Contour vs Ground Truth

In [ ]:
# ── Pitch contour plots for first 2 samples ──────────────────────────

for sid in SAMPLE_IDS[:2]:
    pr = pitch_results[sid]
    gt_notes = gt_data[sid]["midi_notes"]

    midi_curve = pr["midi_clean"]
    voiced = pr["voiced"]
    hop_s = pr["hop_seconds"]

    # Time axis
    t = np.arange(len(midi_curve)) * hop_s

    fig, ax = plt.subplots(figsize=(16, 5))

    # Plot FCPE curve (only voiced frames)
    t_voiced = t[voiced]
    m_voiced = midi_curve[voiced]
    ax.scatter(t_voiced, m_voiced, s=1, alpha=0.5, color="#3498db", label="FCPE (cleaned)")

    # Plot ground truth MIDI notes as horizontal bars
    for note in gt_notes:
        ax.plot(
            [note["start_sec"], note["end_sec"]],
            [note["midi_note"], note["midi_note"]],
            color="#e74c3c", linewidth=3, alpha=0.7,
        )
    # Legend proxy for GT
    ax.plot([], [], color="#e74c3c", linewidth=3, label="Ground Truth (MIDI)")

    # Scoring bands (for context) around GT notes
    for note in gt_notes:
        ax.axhspan(
            note["midi_note"] - 1, note["midi_note"] + 1,
            xmin=(note["start_sec"] - t[0]) / (t[-1] - t[0]),
            xmax=(note["end_sec"] - t[0]) / (t[-1] - t[0]),
            alpha=0.05, color="green",
        )

    # Limit to vocal region
    vocal_times = [n["start_sec"] for n in gt_notes]
    if vocal_times:
        ax.set_xlim(max(0, min(vocal_times) - 2), max(vocal_times) + 5)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("MIDI Note Number")
    ax.set_title(f"Sample {sid} — FCPE Pitch vs Ground Truth MIDI")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

# ── Pitch error histogram ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
errs = np.array(all_pitch_errors)
ax.hist(errs, bins=60, range=(-5, 5), alpha=0.8, color="#3498db", edgecolor="white")
ax.axvline(0, color="black", linestyle="-", linewidth=1)
ax.axvspan(-1, 1, alpha=0.1, color="green", label="±1 st (forgiving perfect)")
ax.axvspan(-2, 2, alpha=0.05, color="orange", label="±2 st (forgiving good)")
ax.set_xlabel("Pitch Error (semitones, + = sharp)")
ax.set_ylabel("Count")
ax.set_title("FCPE Pitch Error Distribution (all samples)")
ax.legend()
plt.tight_layout()
plt.show()

## Section 11: Summary & Recommendations

In [22]:
# ── Final summary table ───────────────────────────────────────────────

print("=" * 70)
print("KARAOKE PIPELINE VALIDATION SUMMARY")
print("=" * 70)

# Alignment
print("\n📐 ALIGNMENT (Qwen3FA vs Whisper)")
print(f"  {'Metric':<25} {'Whisper':>10} {'Qwen3FA':>10}")
print(f"  {'-'*47}")
for key in ["mean_ms", "median_ms", "within_100ms", "within_300ms"]:
    wv = w_report.get(key, "?")
    qv = q_report.get(key, "?")
    unit = "%" if "within" in key else "ms"
    print(f"  {key:<25} {wv:>9}{unit} {qv:>9}{unit}")

winner = "Qwen3FA" if q_report.get("mean_ms", 999) < w_report.get("mean_ms", 999) else "Whisper"
print(f"\n  → Winner: {winner}")

# Pitch
print(f"\n🎵 PITCH ESTIMATION (FCPE)")
abs_all = np.abs(all_pitch_errors)
print(f"  Notes evaluated:       {len(abs_all)}")
print(f"  Mean |error|:          {abs_all.mean():.2f} semitones")
print(f"  Within ±1 semitone:    {(abs_all <= 1.0).mean()*100:.1f}%")
print(f"  Within ±2 semitones:   {(abs_all <= 2.0).mean()*100:.1f}%")

pitch_ok = abs_all.mean() < 2.0
print(f"\n  → FCPE quality: {'GOOD — suitable for forgiving karaoke' if pitch_ok else 'POOR — may need alternatives'}")

# Scoring
print(f"\n🎯 RECOMMENDED SCORING PROFILE")
print(f"  Profile:     'forgiving'")
print(f"  Perfect:     ±1.5 semitones")
print(f"  Good:        ±2.5 semitones")
print(f"  OK:          ±4.0 semitones")
print(f"  Philosophy:  Singer gets benefit of the doubt.")
print(f"               Pipeline errors → free points, not penalties.")

# Cleaning
print(f"\n🧹 CURVE CLEANING")
print(f"  Raw mean |error|:     {raw_abs.mean():.3f} st")
print(f"  Cleaned mean |error|: {clean_abs.mean():.3f} st")
helps = clean_abs.mean() <= raw_abs.mean()
print(f"  → Cleaning {'helps' if helps else 'hurts'} "
      f"(Δ = {raw_abs.mean() - clean_abs.mean():+.3f} st)")

print(f"\n{'='*70}")
print("Tests complete. Run notebook cells sequentially to reproduce.")

KARAOKE PIPELINE VALIDATION SUMMARY

📐 ALIGNMENT (Qwen3FA vs Whisper)
  Metric                       Whisper    Qwen3FA
  -----------------------------------------------
  mean_ms                      5404.6ms    7661.0ms
  median_ms                      46.3ms      37.4ms
  within_100ms                   71.9%      61.8%
  within_300ms                   86.5%      69.4%

  → Winner: Whisper

🎵 PITCH ESTIMATION (FCPE)
  Notes evaluated:       1041
  Mean |error|:          0.87 semitones
  Within ±1 semitone:    73.2%
  Within ±2 semitones:   92.0%

  → FCPE quality: GOOD — suitable for forgiving karaoke

🎯 RECOMMENDED SCORING PROFILE
  Profile:     'forgiving'
  Perfect:     ±1.5 semitones
  Good:        ±2.5 semitones
  OK:          ±4.0 semitones
  Philosophy:  Singer gets benefit of the doubt.
               Pipeline errors → free points, not penalties.

🧹 CURVE CLEANING
  Raw mean |error|:     0.865 st
  Cleaned mean |error|: 0.868 st
  → Cleaning hurts (Δ = -0.002 st)

Tests compl